
# LTIMindtree Azure Data Engineer Interview Experience

**Position:** Data Engineer
**Experience:** 4+ Years

---

## ROUND 1

1. Current project architecture
2. ADF vs Databricks
3. SparkContext vs SparkSession
4. Spark partition logic
5. Data Skew in Spark
6. Cache vs Persist
7. Broadcast Join
8. ROW_NUMBER vs RANK vs DENSE_RANK
9. SQL remove duplicates
10. Types of joins in Spark
11. Delta Lake basics
12. ACID in Delta Lake

---



2. ADF vs Databricks


| **Feature**          | **Azure Data Factory (ADF)**                            | **Azure Databricks**                                                          |
| -------------------- | ------------------------------------------------------- | ----------------------------------------------------------------------------- |
| **Purpose**          | Orchestration, ETL/ELT pipelines                        | Big data analytics, data engineering, machine learning                        |
| **Data Processing**  | Limited; primarily data movement and orchestration      | Advanced distributed data processing using Apache Spark                       |
| **Integration**      | Connects to Azure services and external data sources    | Integrates with Azure services; supports Spark, SQL, Python, Scala, R, and ML |
| **Development**      | GUI-based, low-code/no-code development                 | Notebook-based, code-centric development                                      |
| **Scheduling**       | Built-in triggers, schedules, and event-based execution | Jobs scheduling; can be triggered externally (ADF, APIs, Logic Apps, etc.)    |
| **Monitoring**       | Pipeline execution monitoring, logging, and alerts      | Job, cluster, and notebook execution monitoring                               |
| **Primary Use Case** | Data ingestion, movement, workflow orchestration        | Large-scale data transformation, analytics, streaming, and machine learning   |


3. SparkContext vs SparkSession

| **Feature**                | **SparkContext**                    | **SparkSession**                                                   |
| -------------------------- | ----------------------------------- | ------------------------------------------------------------------ |
| **Purpose**                | Entry point to Spark Core (RDD API) | Unified entry point to all Spark functionalities                   |
| **Introduced In**          | Spark 1.x                           | Spark 2.0                                                          |
| **Primary Use**            | Working with RDDs                   | Working with DataFrames, Datasets, SQL, Streaming, MLlib, and RDDs |
| **API Support**            | RDD API only                        | DataFrame, Dataset, SQL, Streaming, MLlib, and RDD APIs            |
| **SQL Support**            | Requires a separate `SQLContext`    | Built-in SQL support                                               |
| **Hive Support**           | Requires `HiveContext`              | Supports Hive using `.enableHiveSupport()`                         |
| **Ease of Use**            | More components to manage           | Single unified interface                                           |
| **Object Creation**        | `SparkContext()`                    | `SparkSession.builder.getOrCreate()`                               |
| **Access to SparkContext** | Directly available                  | Available through `spark.sparkContext`                             |
| **Recommended**            | Legacy applications                 | Recommended for all new Spark applications                         |

### Code Example

#### SparkContext (Legacy)

```python
from pyspark import SparkContext

sc = SparkContext("local", "MyApp")

rdd = sc.parallelize([1, 2, 3, 4, 5])
print(rdd.collect())
```

#### SparkSession (Recommended)

```python
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .getOrCreate()

df = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
df.show()
```

### Interview Tip

* **SparkContext** is the low-level entry point that initializes the Spark application and is mainly used for **RDD-based processing**.
* **SparkSession** is the **modern, unified entry point** introduced in Spark 2.0. It internally creates a `SparkContext` and provides access to all Spark modules (SQL, DataFrames, Streaming, MLlib, etc.).
* In modern PySpark development, **always use `SparkSession`** unless you're maintaining legacy code.


In [0]:
# 4. Spark partition logic

# In Spark, a partition is the smallest unit of parallel processing. Each partition is processed by one task, and tasks are distributed across executor cores. When reading data, Spark determines partitions based on file size, block size, and configuration settings. During shuffle operations such as joins or aggregations, Spark redistributes data and uses spark.sql.shuffle.partitions (default 200) to decide the number of output partitions. We can manually control partitioning using repartition() to increase or evenly redistribute partitions, and coalesce() to reduce partitions efficiently with minimal shuffle. Proper partitioning is critical for maximizing parallelism, avoiding data skew, and improving Spark job performance.



## 5. What is Data Skew?

**Data skew** occurs when **some Spark partitions contain significantly more data than others**. As a result, some executor tasks finish quickly, while one or a few tasks take much longer because they process a much larger amount of data.

This leads to poor resource utilization and slower overall job execution.

---

## Example

Imagine you're processing employee data partitioned by `department`.

| Department | Number of Records |
| ---------- | ----------------: |
| HR         |               500 |
| Finance    |               800 |
| IT         |             1,200 |
| Sales      |               900 |
| **Others** |     **50,00,000** |

Suppose Spark creates five partitions:

| Partition |       Records |
| --------- | ------------: |
| P1        |           500 |
| P2        |           800 |
| P3        |         1,200 |
| P4        |           900 |
| **P5**    | **50,00,000** |

**What happens?**

* Executors processing **P1–P4** finish quickly.
* The executor processing **P5** keeps running much longer.
* Other executors remain idle, waiting for the last task to complete.

This is called **data skew**.

---

## Why Does Data Skew Happen?

Common causes include:

* Uneven distribution of key values.
* One or a few keys appearing much more frequently than others.
* Skewed joins (for example, joining on a highly repetitive key).
* Grouping or aggregating on skewed columns.
* Poor partitioning strategy.

---

## Impact of Data Skew

* Slower job execution
* Poor cluster utilization
* Long-running tasks (stragglers)
* Increased shuffle time
* Possible executor out-of-memory (OOM) errors
* Higher cloud infrastructure costs

---

## How to Detect Data Skew

### 1. Spark UI

Look for:

* One or a few tasks taking much longer than others.
* Uneven task durations.
* Large shuffle read/write sizes for specific tasks.

### 2. Check Partition Sizes

```python
partition_sizes = df.rdd.glom().map(len).collect()
print(partition_sizes)
```

If one partition is much larger than the others, the data is skewed.

### 3. Count Records by Key

```python
df.groupBy("customer_id").count().orderBy("count", ascending=False).show()
```

If a small number of keys dominate the data, skew is likely.

---

# Techniques to Handle Data Skew

## 1. Repartition Data

Increase or redistribute partitions more evenly.

```python
df = df.repartition(100)
```

**Best for:** General workload balancing.

---

## 2. Salting (Most Common Interview Answer)

Suppose one customer has **5 million** records.

Instead of joining directly:

```text
Customer_ID = 101
```

Add a random salt:

```text
101_0
101_1
101_2
101_3
```

Example:

```python
from pyspark.sql.functions import floor, rand, concat_ws

df = df.withColumn(
    "salted_key",
    concat_ws("_", df.customer_id, floor(rand() * 10))
)
```

This spreads the heavy key across multiple partitions.

**Best for:** Skewed joins.

---

## 3. Broadcast Join

If one table is small, broadcast it to every executor.

```python
from pyspark.sql.functions import broadcast

result = large_df.join(broadcast(small_df), "id")
```

This avoids a large shuffle.

**Best for:** Large table + small lookup table.

---

## 4. Adaptive Query Execution (AQE)

Enable Spark's built-in optimization:

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
```

AQE detects skewed partitions at runtime and splits them into smaller tasks automatically.

---

## 5. Filter Data Early

Remove unnecessary records before joins or aggregations.

```python
filtered_df = df.filter(df.status == "ACTIVE")
```

This reduces the amount of shuffled data.

---

## 6. Bucketing

Store frequently joined tables using buckets on the join key.

```python
df.write.bucketBy(8, "customer_id").saveAsTable("customers")
```

Bucketing can reduce shuffle for repeated joins.

---

# Real-Time Example

An e-commerce company stores **100 million** orders.

* Most customers place fewer than **10 orders**.
* One corporate customer has **20 million orders**.

When joining `orders` with `customers` on `customer_id`, the corporate customer's records end up in one partition, creating a bottleneck.

**Solution:**

* Use a **broadcast join** if `customers` is small.
* If both tables are large, use **salting** or enable **AQE skew join handling**.

---

# Interview Answer (2 Minutes)

> "Data skew occurs when some Spark partitions contain much more data than others. Since each partition is processed by a single task, the larger partitions take much longer to complete, leaving other executors idle. This reduces parallelism and increases job execution time. Data skew commonly occurs during joins and aggregations when certain keys have a disproportionately high number of records. We can identify skew using the Spark UI, partition-size analysis, or key-frequency counts. Common solutions include repartitioning data, using broadcast joins for small tables, salting skewed keys, enabling Adaptive Query Execution (AQE) with skew join optimization, filtering data before shuffles, and bucketing for repeated joins. Proper handling of data skew improves performance, resource utilization, and job stability."


6.  cache() vs persist()
| **Feature**               | **cache()**                                      | **persist()**                                                      |
| ------------------------- | ------------------------------------------------ | ------------------------------------------------------------------ |
| **Purpose**               | Store data for faster reuse                      | Store data with a configurable storage level                       |
| **Default Storage Level** | `MEMORY_AND_DISK` (DataFrame/Dataset)            | User-defined (e.g., `MEMORY_ONLY`, `MEMORY_AND_DISK`, `DISK_ONLY`) |
| **Flexibility**           | No storage level customization                   | Supports multiple storage levels                                   |
| **Performance**           | Faster if data fits in memory                    | Depends on chosen storage level                                    |
| **Disk Storage**          | Uses disk if memory is insufficient (DataFrames) | Optional, based on storage level                                   |
| **Use Case**              | Frequently reused datasets that fit in memory    | Large datasets or when custom storage behavior is needed           |
| **Recommendation**        | Simple and common choice                         | Use when you need control over storage                             |


In [0]:
# 7. Broadcast Join 
# A Broadcast Join is a Spark optimization where the smaller table is sent to every executor, allowing each executor to join it with its local partition of the larger table. This avoids shuffling the large dataset, reduces network overhead, and significantly improves join performance. It's commonly used for fact-to-dimension table joins when the dimension table is small.

8.ROW_NUMBER vs RANK vs DENSE_RANK

| **Feature**          | **ROW_NUMBER()**              | **RANK()**                | **DENSE_RANK()**          |
| -------------------- | ----------------------------- | ------------------------- | ------------------------- |
| **Unique Rank**      | ✅ Yes                         | ❌ No (same rank for ties) | ❌ No (same rank for ties) |
| **Duplicate Values** | Gets different row numbers    | Gets the same rank        | Gets the same rank        |
| **Gap in Ranking**   | ❌ No                          | ✅ Yes                     | ❌ No                      |
| **Best Use Case**    | Remove duplicates, pagination | Competition ranking       | Dense sequential ranking  |


In [0]:
# 9. Remove duplicate value using SQL

WITH CTE AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY Emp_ID
               ORDER BY Emp_ID
           ) AS rn
    FROM Employee
)
DELETE FROM CTE
WHERE rn > 1;

10. Types of joins in Spark

| **Join Type**        | **Description**                                                                     | **Returns**                        |
| -------------------- | ----------------------------------------------------------------------------------- | ---------------------------------- |
| **Inner Join**       | Returns only matching records from both DataFrames                                  | Matching rows only                 |
| **Left Outer Join**  | Returns all rows from the left DataFrame and matching rows from the right           | All left + matched right           |
| **Right Outer Join** | Returns all rows from the right DataFrame and matching rows from the left           | All right + matched left           |
| **Full Outer Join**  | Returns all rows from both DataFrames                                               | Matched + unmatched rows from both |
| **Left Semi Join**   | Returns only rows from the left DataFrame that have a match in the right            | Left rows only (no right columns)  |
| **Left Anti Join**   | Returns only rows from the left DataFrame that do **not** have a match in the right | Non-matching left rows             |
| **Cross Join**       | Returns the Cartesian product of both DataFrames                                    | Every left row × every right row   |


11. Delta Lake basics

# Delta Lake Basics (Interview Explanation)

## What is Delta Lake?

**Delta Lake** is an **open-source storage layer** built on top of a data lake (such as ADLS, S3, or HDFS) that brings **ACID transactions, schema enforcement, versioning, and reliable data processing** to big data workloads.

It stores data in **Parquet format** and maintains a **transaction log (`_delta_log`)** to track all changes.

---

## Why Do We Need Delta Lake?

Traditional data lakes have limitations:

* No ACID transactions
* No version history
* Poor handling of concurrent writes
* No schema enforcement
* Difficult updates and deletes

Delta Lake solves these problems.

---

## Delta Lake Architecture

```text
Application (Spark, Databricks, SQL)
              │
              ▼
        Delta Lake
              │
     ┌────────┴────────┐
     ▼                 ▼
_delta_log         Parquet Files
(Transaction Log)    (Actual Data)
```

* **Parquet files** store the actual data.
* **`_delta_log`** stores transaction history and metadata.

---

## Key Features

| **Feature**            | **Description**                                                       |
| ---------------------- | --------------------------------------------------------------------- |
| **ACID Transactions**  | Ensures reliable and consistent reads/writes                          |
| **Schema Enforcement** | Prevents writing incompatible data                                    |
| **Schema Evolution**   | Allows controlled schema changes                                      |
| **Time Travel**        | Query previous versions of a table                                    |
| **MERGE (Upsert)**     | Insert, update, and delete in a single operation                      |
| **DELETE & UPDATE**    | Modify records directly without rewriting the entire dataset          |
| **Scalable Metadata**  | Uses transaction logs instead of scanning all files                   |
| **Data Versioning**    | Maintains historical versions of data                                 |
| **Concurrent Writes**  | Supports multiple writers safely using optimistic concurrency control |





12. ACID in Delta Lake

# ACID in Delta Lake (Interview Explanation)

**ACID** stands for:

* **A** – Atomicity
* **C** – Consistency
* **I** – Isolation
* **D** – Durability

Delta Lake provides **ACID transactions**, ensuring that data remains accurate and reliable even when multiple users or jobs read and write simultaneously.

---

## 1. Atomicity

**Definition:** A transaction is treated as a single unit—either **all changes succeed** or **none are applied**.

### Example

Suppose you're updating 1,000 employee records.

* If all updates succeed → Changes are committed.
* If the job fails after updating 500 records → **No changes are committed**.

**Result:** Partial updates never occur.

---

## 2. Consistency

**Definition:** Every transaction moves the table from one **valid state** to another, preserving data integrity.

### Example

If a table requires:

```text
Salary > 0
```

Delta Lake prevents invalid writes that would violate the table's schema or constraints (when defined).

**Result:** The data remains valid after every successful transaction.

---

## 3. Isolation

**Definition:** Multiple transactions can run concurrently without interfering with each other.

### Example

* **Job A** updates customer data.
* **Job B** reads customer data at the same time.

Job B sees a **consistent snapshot** of the table rather than partially updated data.

Delta Lake achieves this using **optimistic concurrency control** and **snapshot isolation**.

**Result:** Readers never see incomplete or corrupted data.

---

## 4. Durability

**Definition:** Once a transaction is committed, the data is permanently stored and survives failures.

### Example

A job successfully writes sales data.

Immediately afterward, the cluster crashes.

When the cluster restarts, the committed data is still available because Delta Lake has already recorded the transaction in the **`_delta_log`**.

**Result:** Committed data is not lost.

---






## ROUND 2

1. Azure ETL pipeline design
2. Incremental load strategies
3. Spark fault tolerance
4. Bronze Silver Gold architecture
5. ADF triggers types
6. ADF Integration Runtime
7. ADF pipeline monitoring
8. Pipeline failure handling
9. SQL 2nd highest salary
10. Find duplicate records SQL
11. Partition vs Bucketing
12. Handling late arriving data
13. Spark performance tuning
14. Cluster vs Client mode Spark
15. CDC pipeline design